# Final News Recommender

All features in one notebook:
- **Search cold start** - type any phrase, get closest matching articles
- **Category cold start** - browse by category and subcategory (original method)
- **Negative feedback** - skipped articles nudge the profile away from similar content
- **Popularity signal** - real click data from 156,965 users boosts trending articles
- **Recency boost** - newer articles score slightly higher
- **Diversity injection** - one slot per round explores a different subcategory
- **Mid-session search** - type `s` at any round to pivot to a new topic

**How to run:** Run cells 1-6 in order (setup), then run cell 7 to start.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from collections import Counter
from IPython.display import clear_output

# Load news
news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t", header=None,
    names=["news_id","category","subcategory","title","abstract",
           "url","title_entities","abstract_entities"]
)
news_df["title"]    = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"]      = news_df["url"].fillna("")
news_df = news_df[["news_id","category","subcategory","title","abstract","url"]]
news_df["content"]  = news_df["title"] + " " + news_df["abstract"]

# Load behaviors
behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t", header=None,
    names=["impression_id","user_id","time","history","impressions"]
)
behaviors_df["history"] = behaviors_df["history"].fillna("").apply(lambda x: x.split())

# TF-IDF (fitted once, also used for search queries)
tfidf        = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(news_df["content"])

news_id_to_index         = {nid: idx for idx, nid in enumerate(news_df["news_id"])}
news_df["recency_score"] = news_df.index / len(news_df)

print(f"News loaded      : {len(news_df):,} articles")
print(f"Behaviors loaded : {len(behaviors_df):,} records")
print(f"TF-IDF matrix    : {tfidf_matrix.shape}")


News loaded      : 51,282 articles
Behaviors loaded : 156,965 records
TF-IDF matrix    : (51282, 5000)


In [2]:
# Popularity scores: count real clicks from behaviors, normalise to [0,1]
click_counts = Counter()
behaviors_raw = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t", header=None,
    names=["impression_id","user_id","time","history","impressions"]
)
for imp in behaviors_raw["impressions"]:
    for item in imp.split():
        nid, lbl = item.split("-")
        if int(lbl) == 1:
            click_counts[nid] += 1

news_df["popularity_raw"]   = news_df["news_id"].map(click_counts).fillna(0)
mx                          = news_df["popularity_raw"].max()
news_df["popularity_score"] = news_df["popularity_raw"] / mx if mx > 0 else 0

print(f"Popularity scores computed.")
print(f"Most clicked article: {int(news_df['popularity_raw'].max())} clicks")


Popularity scores computed.
Most clicked article: 4316 clicks


In [3]:
# Core functions

def build_user_profile(clicked_ids):
    """Weighted average of TF-IDF vectors. Recent clicks get higher weight."""
    vectors, weights = [], []
    total = len(clicked_ids)
    for i, nid in enumerate(clicked_ids):
        if nid in news_id_to_index:
            vectors.append(tfidf_matrix[news_id_to_index[nid]])
            weights.append((i + 1) / total)
    if not vectors:
        return None
    stacked = vstack(vectors)
    w = np.array(weights).reshape(-1, 1)
    return np.asarray(stacked.multiply(w).sum(axis=0) / w.sum())


def apply_negative_feedback(profile, skipped_ids, weight=0.05):
    """Subtract a fraction of each skipped article's vector from the profile."""
    if not skipped_ids or profile is None:
        return profile
    adjusted = profile.copy().astype(float)
    for nid in skipped_ids:
        if nid in news_id_to_index:
            adjusted -= weight * np.asarray(
                tfidf_matrix[news_id_to_index[nid]].todense()).flatten()
    return np.clip(adjusted, 0, None)


def recommend(profile, exclude_ids, category=None,
              top_n=5, alpha=0.8, beta=0.1, gamma=0.1, pool=150):
    """
    Score every article by: alpha*similarity + beta*recency + gamma*popularity
    Scan top `pool` similarity candidates first, then re-sort by final score.
    Returns a plain list of dicts (no DataFrame) so indexing is always clean.
    """
    sims    = cosine_similarity(profile, tfidf_matrix).flatten()
    indices = sims.argsort()[::-1]
    cands   = []
    for idx in indices:
        row = news_df.iloc[idx]
        if row["news_id"] in exclude_ids:
            continue
        if category and row["category"] != category:
            continue
        s = float(sims[idx])
        r = float(row["recency_score"])
        p = float(row["popularity_score"])
        cands.append({
            "news_id":    row["news_id"],
            "title":      row["title"],
            "abstract":   str(row["abstract"]),
            "category":   row["category"],
            "subcategory": row["subcategory"],
            "url":        row["url"],
            "sim":        round(s, 4),
            "rec":        round(r, 4),
            "pop":        round(p, 4),
            "score":      round(alpha*s + beta*r + gamma*p, 4),
            "is_diverse": False
        })
        if len(cands) >= pool:
            break
    cands.sort(key=lambda x: x["score"], reverse=True)
    return cands[:top_n]


def add_diversity(cands, exclude_ids):
    """Replace the last slot with an article from a different subcategory."""
    if len(cands) < 2:
        return cands
    dominant_sub = cands[0]["subcategory"]
    shown_ids    = {c["news_id"] for c in cands} | set(exclude_ids)
    pool = news_df[
        (news_df["subcategory"] != dominant_sub) &
        (~news_df["news_id"].isin(shown_ids))
    ].sort_values("popularity_score", ascending=False)
    if pool.empty:
        return cands
    row = pool.iloc[0]
    diverse = {
        "news_id":    row["news_id"],
        "title":      row["title"],
        "abstract":   str(row["abstract"]),
        "category":   row["category"],
        "subcategory": row["subcategory"],
        "url":        row["url"],
        "sim":        0.0,
        "rec":        round(float(row["recency_score"]), 4),
        "pop":        round(float(row["popularity_score"]), 4),
        "score":      0.0,
        "is_diverse": True
    }
    return cands[:-1] + [diverse]


def search_articles(query, top_n=5):
    """
    Transform free-text query with the fitted TF-IDF vectorizer, rank by similarity.
    Returns a plain list of dicts.
    """
    if not query.strip():
        return []
    qvec    = tfidf.transform([query])
    sims    = cosine_similarity(qvec, tfidf_matrix).flatten()
    indices = sims.argsort()[::-1]
    results = []
    for idx in indices:
        if sims[idx] == 0:
            break
        row = news_df.iloc[idx]
        results.append({
            "news_id":    row["news_id"],
            "title":      row["title"],
            "abstract":   str(row["abstract"]),
            "category":   row["category"],
            "subcategory": row["subcategory"],
            "url":        row["url"],
            "sim":        round(float(sims[idx]), 4),
            "rec":        round(float(row["recency_score"]), 4),
            "pop":        round(float(row["popularity_score"]), 4),
            "score":      round(float(sims[idx]), 4),
            "is_diverse": False
        })
        if len(results) >= top_n:
            break
    return results


print("Core functions loaded.")


Core functions loaded.


In [4]:
# UserSession class

class UserSession:
    """Holds all live state for one recommendation session."""

    def __init__(self):
        self.clicked_ids   = []    # articles added to history (clicks + seeds)
        self.profile       = None  # current TF-IDF profile vector
        self.total_skipped = 0     # cumulative skips penalised
        self.round         = 0
        self.log           = []    # one entry per round

    def seed_from_articles(self, article_list):
        """Build initial profile from a list of article dicts."""
        self.clicked_ids = [a["news_id"] for a in article_list]
        self.profile     = build_user_profile(self.clicked_ids)

    def click(self, news_id, skipped_ids, neg_weight=0.05):
        """Record a click, rebuild profile, apply negative feedback for skips."""
        if news_id not in self.clicked_ids:
            self.clicked_ids.append(news_id)
        self.profile = build_user_profile(self.clicked_ids)
        if skipped_ids:
            self.total_skipped += len(skipped_ids)
            self.profile = apply_negative_feedback(
                self.profile, skipped_ids, neg_weight
            )

    def dominant_category(self):
        rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        return rows["category"].value_counts().idxmax() if not rows.empty else "unknown"

    def interest_summary(self):
        rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        return rows["category"].value_counts().to_dict()

    def log_round(self, shown, clicked, skipped):
        self.log.append({
            "round":    self.round,
            "shown":    shown,
            "clicked":  clicked,
            "skipped":  skipped,
            "dominant": self.dominant_category()
        })


print("UserSession loaded.")


UserSession loaded.


In [5]:
# Display functions

DIVIDER = "=" * 63
LINE    = "-" * 63


def show_articles(articles, header):
    """
    Print a numbered list of articles.
    Works for recommendations, search results, and seed articles.
    articles is always a plain list of dicts so [i+1] is always correct.
    """
    print("\n" + DIVIDER)
    print("  " + header)
    print(DIVIDER)
    for i, a in enumerate(articles):
        tag      = "  [EXPLORE]" if a.get("is_diverse") else ""
        title    = a["title"][:66] + "..." if len(a["title"]) > 66 else a["title"]
        abstract = a["abstract"][:88] + "..." if len(a["abstract"]) > 88 else a["abstract"]
        print(f"\n  [{i+1}] {title}{tag}")
        print(f"       {a['category']} > {a['subcategory']}")
        if a["score"] > 0:
            print(f"       score={a['score']:.4f}  "
                  f"(sim={a['sim']:.3f}, rec={a['rec']:.3f}, pop={a['pop']:.3f})")
        print(f"       {abstract}")
    print("\n" + LINE)


def show_profile(session, skipped_titles):
    """Show profile drift after each click."""
    summary  = session.interest_summary()
    dominant = session.dominant_category()
    print(f"\n  Profile updated:")
    print(f"  History: {len(session.clicked_ids)} articles  |  "
          f"Skips penalised: {session.total_skipped}  |  "
          f"Dominant topic: {dominant}")
    if skipped_titles:
        print("\n  Down-weighted this round (skipped):")
        for t in skipped_titles:
            short = t[:62] + "..." if len(t) > 62 else t
            print(f"    - {short}")
    print("\n  Interest breakdown:", end="")
    for cat, cnt in sorted(summary.items(), key=lambda x: x[1], reverse=True):
        bar = "#" * cnt
        print(f"\n    {cat:<22} {bar} ({cnt})", end="")
    print("\n")


def show_summary(session, cold_start_label):
    """End-of-session summary."""
    print("\n" + DIVIDER)
    print("  SESSION SUMMARY")
    print(DIVIDER)
    print(f"  Started with     : {cold_start_label}")
    print(f"  Rounds completed : {session.round}")
    print(f"  Articles read    : {len(session.clicked_ids)}")
    print(f"  Skips penalised  : {session.total_skipped}")
    print(f"  Dominant topic   : {session.dominant_category()}")
    print("\n  Round-by-round:")
    for entry in session.log:
        rows  = news_df[news_df["news_id"] == entry["clicked"]]
        title = ""
        if not rows.empty:
            t = rows.iloc[0]["title"]
            title = t[:52] + "..." if len(t) > 52 else t
        print(f"  Round {entry['round']:>2}  |  {title}")
        print(f"           |  Skipped {len(entry['skipped'])} articles  "
              f"|  Dominant: {entry['dominant']}")
    print("\n  Categories explored:")
    for cat, cnt in sorted(session.interest_summary().items(),
                           key=lambda x: x[1], reverse=True):
        print(f"    {cat:<25} {cnt} articles")
    print("\n" + DIVIDER)
    print("  Thanks for using the News Recommender!")
    print(DIVIDER + "\n")


print("Display functions loaded.")


Display functions loaded.


In [6]:
# run_session() - main interactive loop

def run_session(top_n=5, alpha=0.8, beta=0.1, gamma=0.1, neg_weight=0.05):
    """
    Full interactive recommendation session.

    Cold start:
      [1] Search  - type any phrase, get closest matching articles automatically
      [2] Browse  - pick a category then subcategory

    During the session:
      1 to 5  - read that article (updates your profile)
      s       - search for a topic to pivot your session mid-way
      q       - quit and see session summary
    """

    session = UserSession()

    # ==========================================================================
    # STEP 1: Choose how to start
    # ==========================================================================
    print("\n" + DIVIDER)
    print("  WELCOME TO THE NEWS RECOMMENDER")
    print(DIVIDER)
    print("\n  How would you like to start?\n")
    print("  [1]  Search for a topic")
    print("       Type anything in plain English - 'home finance', 'NBA', 'climate change'")
    print("       The system finds the closest matching articles automatically.\n")
    print("  [2]  Browse by category")
    print("       Choose from a list of categories and subcategories.\n")

    while True:
        method = input("  Enter 1 or 2: ").strip()
        if method in ("1", "2"):
            break
        print("  Please enter 1 or 2.")

    # ==========================================================================
    # STEP 2a: Search cold start
    # ==========================================================================
    if method == "1":
        print("\n  Type what kind of news you want.")
        print("  Examples: 'NBA basketball', 'stock market', 'healthy recipes'\n")

        seed_articles  = []
        cold_start_label = ""

        while True:
            query = input("  Your search query: ").strip()
            if not query:
                print("  Please type something.")
                continue

            results = search_articles(query, top_n=top_n)

            if not results:
                print("  No matching articles found. Try different keywords.\n")
                continue

            clear_output(wait=True)
            show_articles(results, "Matching articles for: '" + query + "'")
            confirm = input(
                "  Press Enter to start with these articles, "
                "or type a new query to search again: "
            ).strip()

            if not confirm:
                seed_articles    = results
                cold_start_label = "Search: '" + query + "'"
                break
            # User typed a new query - use it on the next loop
            query = confirm

        session.seed_from_articles(seed_articles)
        cat_filter = None   # search can span multiple categories

    # ==========================================================================
    # STEP 2b: Category cold start
    # ==========================================================================
    else:
        categories = sorted(news_df["category"].unique())
        print("\n  Available categories:\n")
        for i, cat in enumerate(categories):
            print(f"    {i:>2}: {cat}")

        while True:
            try:
                ci = int(input("\n  Select category number: "))
                if 0 <= ci < len(categories):
                    break
                print(f"  Please enter a number between 0 and {len(categories)-1}.")
            except ValueError:
                print("  Please enter a number.")

        selected_cat = categories[ci]
        subcats = sorted(
            news_df[news_df["category"] == selected_cat]["subcategory"].unique()
        )
        print(f"\n  Subcategories for '{selected_cat}':\n")
        for i, sub in enumerate(subcats):
            print(f"    {i:>2}: {sub}")

        while True:
            try:
                si = int(input("\n  Select subcategory number: "))
                if 0 <= si < len(subcats):
                    break
                print(f"  Please enter a number between 0 and {len(subcats)-1}.")
            except ValueError:
                print("  Please enter a number.")

        selected_sub = subcats[si]

        # Seed from top articles in chosen subcategory
        seed_rows = news_df[
            (news_df["category"]    == selected_cat) &
            (news_df["subcategory"] == selected_sub)
        ].head(top_n)

        # Fallback to category level if subcategory is too small
        if len(seed_rows) < 2:
            print(f"  Not enough articles in subcategory, using full '{selected_cat}' category.")
            seed_rows = news_df[news_df["category"] == selected_cat].head(top_n)

        seed_articles = []
        for _, row in seed_rows.iterrows():
            seed_articles.append({
                "news_id":    row["news_id"],
                "title":      row["title"],
                "abstract":   str(row["abstract"]),
                "category":   row["category"],
                "subcategory": row["subcategory"],
                "url":        row["url"],
                "sim": 0.0, "rec": 0.0, "pop": 0.0,
                "score": 0.0, "is_diverse": False
            })

        session.seed_from_articles(seed_articles)
        cold_start_label = "Category: " + selected_cat + " > " + selected_sub
        cat_filter = selected_cat   # stay within chosen category

    # ==========================================================================
    # STEP 3: Main recommendation loop
    # ==========================================================================
    while True:
        session.round += 1

        # Generate recommendations
        recs = recommend(
            session.profile, session.clicked_ids,
            category=cat_filter,
            top_n=top_n, alpha=alpha, beta=beta, gamma=gamma
        )

        # Fallback: category pool exhausted, open to all categories
        if not recs:
            cat_filter = None
            recs = recommend(
                session.profile, session.clicked_ids,
                top_n=top_n, alpha=alpha, beta=beta, gamma=gamma
            )

        # Add one diversity article into the last slot
        recs = add_diversity(recs, session.clicked_ids)

        # Show recommendations - clear previous output first
        clear_output(wait=True)
        show_articles(recs, f"Round {session.round} - Top Recommendations")

        nw_info = f"neg_weight={neg_weight}" if neg_weight > 0 else "negative feedback off"
        print(f"  Enter 1-{len(recs)} to read  |  s to search  |  q to quit  |  [{nw_info}]")
        choice = input("\n  Your choice: ").strip().lower()

        # ------------------------------------------------------------------
        # QUIT
        # ------------------------------------------------------------------
        if choice == "q":
            show_summary(session, cold_start_label)
            break

        # ------------------------------------------------------------------
        # MID-SESSION SEARCH PIVOT
        # ------------------------------------------------------------------
        if choice == "s":
            sq = input("  Search query: ").strip()
            if not sq:
                print("  Empty query - going back.")
                session.round -= 1
                continue

            sr = search_articles(sq, top_n=top_n)
            if not sr:
                print("  No results found. Going back.")
                session.round -= 1
                continue

            clear_output(wait=True)
            show_articles(sr, "Search results: '" + sq + "'")
            print(f"  Enter 1-{len(sr)} to read  |  Press Enter to go back")
            sc = input("\n  Your choice: ").strip()

            if not sc:
                session.round -= 1
                continue

            try:
                si = int(sc) - 1
                if not (0 <= si < len(sr)):
                    raise ValueError
            except ValueError:
                print("  Invalid choice - going back.")
                session.round -= 1
                continue

            picked = sr[si]
            print(f"\n  You selected : {picked['title'][:70]}")
            print(f"  Category     : {picked['category']} > {picked['subcategory']}")

            session.log_round(
                shown   = [a["news_id"] for a in sr],
                clicked = picked["news_id"],
                skipped = []
            )
            session.click(picked["news_id"], skipped_ids=[], neg_weight=0)
            show_profile(session, [])
            continue

        # ------------------------------------------------------------------
        # NORMAL ARTICLE SELECTION (1 to top_n)
        # ------------------------------------------------------------------
        try:
            ci = int(choice) - 1
            if not (0 <= ci < len(recs)):
                raise ValueError
        except ValueError:
            print(f"  Invalid input. Please enter 1-{len(recs)}, 's', or 'q'.")
            session.round -= 1
            continue

        picked         = recs[ci]
        clicked_id     = picked["news_id"]
        skipped_ids    = [a["news_id"] for a in recs if a["news_id"] != clicked_id]
        skipped_titles = [a["title"]   for a in recs if a["news_id"] != clicked_id]

        print(f"\n  You selected : {picked['title'][:70]}")
        print(f"  Category     : {picked['category']} > {picked['subcategory']}")
        print(f"  Penalising   : {len(skipped_ids)} skipped articles")

        session.log_round(
            shown   = [a["news_id"] for a in recs],
            clicked = clicked_id,
            skipped = skipped_ids
        )
        session.click(clicked_id, skipped_ids=skipped_ids, neg_weight=neg_weight)
        show_profile(session, skipped_titles)


print("run_session() ready.")


run_session() ready.


In [ ]:
# Run this cell to start
#
# You will be asked to choose:
#   [1]  Search for a topic
#        Type anything in plain English to find matching articles automatically
#
#   [2]  Browse by category
#        Pick from a list of categories and subcategories
#
# During the session:
#   1-5  -> read that article (profile updates automatically)
#   s    -> search for any topic to steer your session
#   q    -> quit and see your full session summary

run_session(
    top_n      = 5,      # articles shown per round
    alpha      = 0.8,    # similarity weight
    beta       = 0.1,    # recency weight
    gamma      = 0.1,    # popularity weight
    neg_weight = 0.05    # skip penalty strength (set to 0.0 to turn off)
)



  Matching articles for: 'Kanye'

  [1] Kanye West says he'll never perform old song lyrics again
       music > musicnews
       score=0.6434  (sim=0.643, rec=0.462, pop=0.000)
       Kanye says he's playing music for God now.

  [2] Kanye West Disses DWTS in New Song   and Val Chmerkovskiy Includes...
       entertainment > celebrity
       score=0.6200  (sim=0.620, rec=0.460, pop=0.000)
       Val Chmerkovskiy Responds to Kanye West's DWTS Diss

  [3] Kanye West says he's coming to Houston for Sunday church service
       music > musicnews
       score=0.5578  (sim=0.558, rec=0.809, pop=0.002)
       Well, it looks like Joel Osteen prayers have been answered! About two weeks ago, he exte...

  [4] Celebs Who Have Rocked a Mullet, From Meryl Streep to Kanye West
       entertainment > celebrity
       score=0.5545  (sim=0.554, rec=0.079, pop=0.000)
       Business in the front, party in the back.

  [5] Kanye West requested those working on 'Jesus is King' to 'fast,' '...
       mus